# 05 — Whole-Gland Radiomic Feature Extraction

## Objective

This notebook extracts patient-level radiomic features from the original
T2W MRI volumes using the 3D whole-gland prostate masks predicted by the
trained U-Net.

The workflow is:

1. Configure the radiomics environment and project paths
2. Configure a reproducible PyRadiomics extractor
3. Validate the complete workflow on one MRI examination
4. Extract radiomic features for the complete dataset
5. Validate and save the final patient-level feature table

The resulting feature table will be used in the classification stage.

This variant uses the grouped-v2 predictions and writes separate outputs, preserving the original results.


## 1. Environment and Paths

This notebook uses the dedicated Python 3.8 radiomics environment.

The original T2W MRI volumes are read directly from the PI-CAI dataset,
while the segmentation masks are read from the U-Net inference results.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import SimpleITK as sitk
import radiomics

from radiomics import featureextractor

PROJECT_ROOT = Path.cwd().parent

clinical_path = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "raw"
    / "picai_labels"
    / "clinical_information"
    / "marksheet.csv"
)

picai_dir = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "raw"
    / "picai"
)

predicted_masks_dir = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "processed"
    / "whole_gland_predictions_grouped_v2"
)

features_output_dir = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "features"
)

features_output_path = (
    features_output_dir
    / "whole_gland_t2w_radiomics_grouped_v2.csv"
)

failures_output_path = (
    features_output_dir
    / "whole_gland_t2w_radiomics_grouped_v2_failures.csv"
)

features_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

clinical_dataframe = pd.read_csv(
    clinical_path,
    dtype={
        "patient_id": str,
        "study_id": str,
    },
)

# Confere a correspondência entre os exames clínicos e as máscaras grouped_v2.
expected_cases = set(
    clinical_dataframe["patient_id"].astype(str)
    + "_"
    + clinical_dataframe["study_id"].astype(str)
)
available_cases = {
    p.name[:-len("_whole_gland_pred.mha")]
    for p in predicted_masks_dir.glob("*_whole_gland_pred.mha")
}
missing_cases = expected_cases - available_cases
extra_cases = available_cases - expected_cases

print("Cases expected:", len(expected_cases))
print("Masks found:", len(available_cases))
print("Missing masks:", len(missing_cases))
print("Extra masks:", len(extra_cases))

if len(expected_cases) != 1500 or missing_cases or extra_cases:
    raise ValueError(
        "Grouped-v2 masks do not match all 1,500 clinical examinations. "
        f"Missing examples: {sorted(missing_cases)[:5]}, "
        f"extra examples: {sorted(extra_cases)[:5]}"
    )

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SimpleITK:", sitk.Version_VersionString())
print("PyRadiomics:", radiomics.__version__)
print("Clinical cases:", len(clinical_dataframe))
print("Predicted masks:", len(list(predicted_masks_dir.glob("*.mha"))))


Cases expected: 1500
Masks found: 1500
Missing masks: 0
Extra masks: 0
Python: 3.8.10
NumPy: 1.24.4
Pandas: 2.0.3
SimpleITK: 2.3.1
PyRadiomics: v3.1.0
Clinical cases: 1500
Predicted masks: 1500


## 2. Final Radiomics Configuration

The extraction protocol is fixed before processing the complete dataset.

The configuration uses the original T2W image, 3D whole-gland segmentation,
intensity normalization, fixed-width discretization, and standardized voxel
spacing. Only the standard PyRadiomics feature classes are enabled.


In [2]:
extractor = featureextractor.RadiomicsFeatureExtractor(
    normalize=True,
    normalizeScale=100,
    binWidth=5,
    resampledPixelSpacing=[
        0.5,
        0.5,
        3.0,
    ],
    interpolator=sitk.sitkBSpline,
)

extractor.disableAllImageTypes()
extractor.enableImageTypeByName("Original")

extractor.disableAllFeatures()

for feature_class in [
    "firstorder",
    "shape",
    "glcm",
    "glrlm",
    "glszm",
    "gldm",
    "ngtdm",
]:
    extractor.enableFeatureClassByName(
        feature_class
    )

print("Normalize:", extractor.settings["normalize"])
print("Normalize scale:", extractor.settings["normalizeScale"])
print("Bin width:", extractor.settings["binWidth"])
print(
    "Resampled spacing:",
    extractor.settings["resampledPixelSpacing"],
)
print("Image types:", extractor.enabledImagetypes)
print("Feature classes:", extractor.enabledFeatures)


Normalize: True
Normalize scale: 100
Bin width: 5
Resampled spacing: [0.5, 0.5, 3.0]
Image types: {'Original': {}}
Feature classes: {'firstorder': [], 'shape': [], 'glcm': [], 'glrlm': [], 'glszm': [], 'gldm': [], 'ngtdm': []}


## 3. Single-Case Validation

One MRI examination is processed before batch extraction.

This step verifies file availability, image-mask geometry, mask validity,
feature count, and numerical validity of the extracted features.


In [3]:
def get_case_paths(
    patient_id,
    study_id,
):
    t2w_path = (
        picai_dir
        / patient_id
        / f"{patient_id}_{study_id}_t2w.mha"
    )

    mask_path = (
        predicted_masks_dir
        / f"{patient_id}_{study_id}_whole_gland_pred.mha"
    )

    return t2w_path, mask_path


def validate_image_and_mask(
    image,
    mask,
    mask_array,
):
    same_size = (
        image.GetSize()
        == mask.GetSize()
    )

    same_spacing = np.allclose(
        image.GetSpacing(),
        mask.GetSpacing(),
        atol=1e-6,
    )

    same_origin = np.allclose(
        image.GetOrigin(),
        mask.GetOrigin(),
        atol=1e-6,
    )

    same_direction = np.allclose(
        image.GetDirection(),
        mask.GetDirection(),
        atol=1e-6,
    )

    mask_values = set(
        np.unique(mask_array).tolist()
    )

    binary_mask = mask_values.issubset(
        {0, 1}
    )

    non_empty_mask = (
        int(mask_array.sum()) > 0
    )

    if not all(
        [
            same_size,
            same_spacing,
            same_origin,
            same_direction,
            binary_mask,
            non_empty_mask,
        ]
    ):
        raise ValueError(
            "Image-mask validation failed."
        )


def extract_radiomic_features(
    image,
    mask,
):
    result = extractor.execute(
        image,
        mask,
    )

    features = {
        name: float(value)
        for name, value in result.items()
        if not name.startswith(
            "diagnostics_"
        )
    }

    values = np.asarray(
        list(features.values()),
        dtype=np.float64,
    )

    if len(features) != 107:
        raise ValueError(
            f"Unexpected feature count: {len(features)}"
        )

    if not np.isfinite(values).all():
        raise ValueError(
            "Non-finite radiomic feature detected."
        )

    return features


In [4]:
patient_id = "10001"
study_id = "1000001"

t2w_path, mask_path = get_case_paths(
    patient_id,
    study_id,
)

if not t2w_path.exists():
    raise FileNotFoundError(
        f"T2W image not found: {t2w_path}"
    )

if not mask_path.exists():
    raise FileNotFoundError(
        f"Predicted mask not found: {mask_path}"
    )

t2w_image = sitk.ReadImage(
    str(t2w_path)
)

mask_image = sitk.ReadImage(
    str(mask_path)
)

mask_array = sitk.GetArrayFromImage(
    mask_image
)

validate_image_and_mask(
    image=t2w_image,
    mask=mask_image,
    mask_array=mask_array,
)

single_case_features = extract_radiomic_features(
    image=t2w_image,
    mask=mask_image,
)

single_case_values = np.asarray(
    list(single_case_features.values()),
    dtype=np.float64,
)

print(
    "Case:",
    f"{patient_id}_{study_id}",
)
print(
    "Image size:",
    t2w_image.GetSize(),
)
print(
    "Image spacing:",
    t2w_image.GetSpacing(),
)
print(
    "Positive mask voxels:",
    int(mask_array.sum()),
)
print(
    "Radiomic features:",
    len(single_case_features),
)
print(
    "NaN values:",
    int(np.isnan(single_case_values).sum()),
)
print(
    "Infinite values:",
    int(np.isinf(single_case_values).sum()),
)


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Case: 10001_1000001
Image size: (640, 640, 21)
Image spacing: (0.30000001192093, 0.30000001192093, 3.6000000778884846)
Positive mask voxels: 279860
Radiomic features: 107
NaN values: 0
Infinite values: 0


## 4. Batch Radiomic Feature Extraction

The validated procedure is applied to all MRI examinations.

The extraction supports resume from an existing output file and saves
periodic checkpoints. Failed cases are recorded separately without stopping
the complete batch.


In [5]:
CHECKPOINT_INTERVAL = 25

if features_output_path.exists():
    features_dataframe = pd.read_csv(
        features_output_path,
        dtype={
            "patient_id": str,
            "study_id": str,
        },
    )

    # Não retomar um arquivo incompatível ou com exames não pertencentes ao dataset.
    existing_columns = set(features_dataframe.columns)
    if not {"patient_id", "study_id"}.issubset(existing_columns):
        raise ValueError("Existing grouped-v2 file lacks case identifiers.")
    if len(existing_columns) != 109:
        raise ValueError("Existing grouped-v2 file must have 107 features and 2 IDs.")
    existing_ids = set(
        features_dataframe["patient_id"].astype(str)
        + "_" + features_dataframe["study_id"].astype(str)
    )
    if len(existing_ids) != len(features_dataframe) or not existing_ids <= expected_cases:
        raise ValueError("Existing grouped-v2 file contains duplicate or unknown cases.")

    processed_case_ids = set(
        features_dataframe[
            "patient_id"
        ].astype(str)
        + "_"
        + features_dataframe[
            "study_id"
        ].astype(str)
    )

    feature_records = (
        features_dataframe
        .to_dict(
            orient="records"
        )
    )

    print(
        "Resuming from existing file:",
        len(feature_records),
        "cases",
    )
else:
    processed_case_ids = set()
    feature_records = []

failed_records = []

for index, row in clinical_dataframe.iterrows():
    patient_id = str(
        row["patient_id"]
    )
    study_id = str(
        row["study_id"]
    )
    case_id = (
        f"{patient_id}_{study_id}"
    )

    if case_id in processed_case_ids:
        continue

    print(
        f"[{index + 1}/{len(clinical_dataframe)}] "
        f"Extracting {case_id}"
    )

    try:
        t2w_path, mask_path = get_case_paths(
            patient_id,
            study_id,
        )

        if not t2w_path.exists():
            raise FileNotFoundError(
                f"T2W image not found: {t2w_path}"
            )

        if not mask_path.exists():
            raise FileNotFoundError(
                f"Predicted mask not found: {mask_path}"
            )

        t2w_image = sitk.ReadImage(
            str(t2w_path)
        )

        mask_image = sitk.ReadImage(
            str(mask_path)
        )

        mask_array = sitk.GetArrayFromImage(
            mask_image
        )

        validate_image_and_mask(
            image=t2w_image,
            mask=mask_image,
            mask_array=mask_array,
        )

        case_features = extract_radiomic_features(
            image=t2w_image,
            mask=mask_image,
        )

        record = {
            "patient_id": patient_id,
            "study_id": study_id,
        }

        record.update(
            case_features
        )

        feature_records.append(
            record
        )

        processed_case_ids.add(
            case_id
        )

        if (
            len(feature_records)
            % CHECKPOINT_INTERVAL
            == 0
        ):
            checkpoint_dataframe = pd.DataFrame(
                feature_records
            )

            checkpoint_dataframe.to_csv(
                features_output_path,
                index=False,
            )

            print(
                "Checkpoint saved:",
                len(feature_records),
                "cases",
            )

    except Exception as error:
        failed_records.append(
            {
                "patient_id": patient_id,
                "study_id": study_id,
                "case_id": case_id,
                "error": str(error),
            }
        )

        print(
            f"Failed {case_id}: {error}"
        )

features_dataframe = pd.DataFrame(
    feature_records
)

features_dataframe.to_csv(
    features_output_path,
    index=False,
)

failures_dataframe = pd.DataFrame(
    failed_records
)

failures_dataframe.to_csv(
    failures_output_path,
    index=False,
)

print(
    "Successful cases:",
    len(features_dataframe),
)

print(
    "Failed cases:",
    len(failures_dataframe),
)


[1/1500] Extracting 10000_1000000


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


[2/1500] Extracting 10001_1000001


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


[3/1500] Extracting 10002_1000002


...

[1499/1500] Extracting 11474_1001498


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


[1500/1500] Extracting 11475_1001499


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Checkpoint saved: 1500 cases
Successful cases: 1500
Failed cases: 0


## 5. Final Dataset Validation

The final feature table is checked before it is used by the classification
pipeline.

Each successful MRI examination must correspond to one row containing two
identifiers and 107 radiomic features.


In [6]:
features_dataframe = pd.read_csv(
    features_output_path,
    dtype={
        "patient_id": str,
        "study_id": str,
    },
)

identifier_columns = [
    "patient_id",
    "study_id",
]

feature_columns = [
    column
    for column in features_dataframe.columns
    if column not in identifier_columns
]

feature_matrix = features_dataframe[
    feature_columns
].to_numpy(
    dtype=np.float64
)

duplicated_cases = (
    features_dataframe
    .duplicated(
        subset=identifier_columns
    )
    .sum()
)

print(
    "Final table shape:",
    features_dataframe.shape,
)

print(
    "Cases:",
    len(features_dataframe),
)

print(
    "Radiomic features:",
    len(feature_columns),
)

print(
    "Duplicated cases:",
    int(duplicated_cases),
)

print(
    "NaN values:",
    int(
        np.isnan(
            feature_matrix
        ).sum()
    ),
)

print(
    "Infinite values:",
    int(
        np.isinf(
            feature_matrix
        ).sum()
    ),
)

print(
    "Feature file:",
    features_output_path,
)

print(
    "Failure file:",
    failures_output_path,
)


Final table shape: (1500, 109)
Cases: 1500
Radiomic features: 107
Duplicated cases: 0
NaN values: 0
Infinite values: 0
Feature file: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\features\whole_gland_t2w_radiomics_grouped_v2.csv
Failure file: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\features\whole_gland_t2w_radiomics_grouped_v2_failures.csv


## 6. Summary

The original T2W MRI volumes were combined with the predicted 3D
whole-gland prostate masks to generate a patient-level radiomic dataset.

The final table contains the patient and study identifiers together with
107 PyRadiomics features extracted from each successfully processed MRI
examination.

This table is the input for the subsequent machine-learning classification
stage.
